# 01 · E1 零样本冒烟（5 样本）

**目的**：在 Kaggle 上验证三件事：

1. `open_clip` 能加载 ViT-B-16，并暴露 patch tokens（兼容 timm `trunk` 与 `forward_intermediates` 两种实现）；
2. E1 零样本相似度热图能跑通并可视化；
3. IoU / NSS 指标函数在真实样本上工作正常。

> 这是**冒烟测试**，结果不写进论文。正式实验使用 AGD20K 的 Seen / Easy-Unseen / Hard-Unseen 协议。
> 注意：HF 子集的掩码是围绕交互点的粗粒度热图块，不是 AGD20K 的精细部件掩码。

## 准备

1. 把 `affordance-smoke5.zip` 上传为**私有 Kaggle Dataset**（名称 `affordance-smoke5`）；
2. Notebook 右侧 **Add Input** → 选择该 Dataset；
3. Settings：Accelerator = **GPU T4 ×2**（CPU 也能跑，只是慢），Internet = **On**；
4. Run All，预计 5–10 分钟（含下载 CLIP 权重）。

> 若 CLIP 权重下载失败，在安装 cell 前先执行：`import os; os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'`。

In [ ]:
import sys, importlib.metadata as md
print(sys.version)
for pkg in ['torch', 'torchvision', 'numpy', 'pandas', 'matplotlib', 'pillow']:
    try:
        print(pkg, md.version(pkg))
    except Exception as exc:
        print(pkg, 'MISSING', exc)
import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q open_clip_torch timm
import open_clip, timm, importlib.metadata as md
print('open_clip_torch', md.version('open_clip_torch'))
print('timm', md.version('timm'))

In [ ]:
from pathlib import Path
import csv

search_roots = [Path('/kaggle/input')] if Path('/kaggle/input').exists() else []
search_roots += [Path.cwd(), *Path.cwd().parents]

meta_candidates = []
for root in search_roots:
    if root.exists():
        meta_candidates.extend(root.rglob('smoke_meta.csv'))
if not meta_candidates:
    raise FileNotFoundError('未找到 smoke_meta.csv：请把 affordance-smoke5 添加为 Notebook 输入')

META = meta_candidates[0]
ROOT = META.parent
rows = list(csv.DictReader(open(META, encoding='utf-8-sig')))
print('meta:', META)
print('samples:', len(rows))
for r in rows:
    print(' -', r['object'], '/', r['affordance'], '|', r['image'])

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

IMAGE_SIZE = 224

def load_sample(row, size=IMAGE_SIZE):
    img = Image.open(ROOT / row['image']).convert('RGB').resize((size, size), Image.Resampling.BICUBIC)
    mask = Image.open(ROOT / row['mask']).convert('L').resize((size, size), Image.Resampling.NEAREST)
    return np.asarray(img), (np.asarray(mask) > 127).astype(np.float32)

fig, axes = plt.subplots(len(rows), 3, figsize=(9, 3 * len(rows)))
for i, row in enumerate(rows):
    img, mask = load_sample(row)
    overlay = img.copy()
    overlay[mask > 0.5] = (overlay[mask > 0.5] * 0.4 + np.array([220, 40, 40]) * 0.6).astype(np.uint8)
    axes[i, 0].imshow(img); axes[i, 0].set_title(f"{row['object']} / {row['affordance']}", fontsize=9)
    axes[i, 1].imshow(mask, cmap='gray'); axes[i, 1].set_title('GT mask', fontsize=9)
    axes[i, 2].imshow(overlay); axes[i, 2].set_title('GT overlay', fontsize=9)
    for ax in axes[i]:
        ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import torch
import open_clip
from torchvision import transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _, _ = open_clip.create_model_and_transforms('ViT-B-16', pretrained='laion2b_s34b_b88k')
tokenizer = open_clip.get_tokenizer('ViT-B-16')
model = model.to(device).eval()
for p in model.parameters():
    p.requires_grad = False

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])

def tokens_to_grid(tokens):
    b, n, d = tokens.shape
    side = int(round(n ** 0.5))
    assert side * side == n, f'patch 数 {n} 不是完全平方数'
    return tokens.transpose(1, 2).reshape(b, d, side, side), side

def patch_grid(images):
    visual = model.visual

    # 1) timm 视觉塔
    trunk = getattr(visual, 'trunk', None)
    if trunk is not None and hasattr(trunk, 'forward_features'):
        feats = trunk.forward_features(images)
        if isinstance(feats, dict):
            feats = feats.get('x')
        n_prefix = int(getattr(trunk, 'num_prefix_tokens', 1))
        return tokens_to_grid(feats[:, n_prefix:, :])

    # 2) 自定义 VisionTransformer（open_clip >= 2.32）
    if hasattr(visual, 'forward_intermediates'):
        out = visual.forward_intermediates(
            images, indices=1, normalize_intermediates=True,
            intermediates_only=True, output_fmt='NCHW',
        )
        grid = out['image_intermediates'][-1]
        side = int(round((grid.shape[-2] * grid.shape[-1]) ** 0.5))
        return grid, side

    # 3) 旧版 CLIPVisionTransformer
    if hasattr(visual, 'conv1') and hasattr(visual, 'transformer') and hasattr(visual, 'class_embedding'):
        x = visual.conv1(images)
        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
        cls = visual.class_embedding.to(x.dtype) + torch.zeros(
            x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device
        )
        x = torch.cat([cls, x], dim=1) + visual.positional_embedding.to(x.dtype)
        x = visual.ln_pre(x).permute(1, 0, 2)
        x = visual.transformer(x)
        if isinstance(x, (tuple, list)):
            x = x[0]
        return tokens_to_grid(x.permute(1, 0, 2)[:, 1:, :])

    raise RuntimeError('无法从当前 open_clip 视觉塔取 patch 特征')

dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
with torch.no_grad():
    grid, side = patch_grid(dummy)
print('patch grid:', tuple(grid.shape), '| side:', side)

In [ ]:
import torch.nn.functional as F

PROMPT_TEMPLATE = 'the part of a {object} you can {affordance}'

def build_prompt(obj, aff):
    return PROMPT_TEMPLATE.format(object=obj.replace('_', ' '), affordance=aff.replace('_', ' '))

@torch.no_grad()
def e1_heatmap(rows, size=IMAGE_SIZE):
    images = torch.stack([
        image_transform(Image.open(ROOT / r['image']).convert('RGB')) for r in rows
    ]).to(device)
    prompts = [build_prompt(r['object'], r['affordance']) for r in rows]
    grid, side = patch_grid(images)
    text = model.encode_text(tokenizer(prompts).to(device))
    b, d, _, _ = grid.shape
    flat = F.normalize(grid.reshape(b, d, side * side), dim=1)
    text = F.normalize(text, dim=-1)
    sim = torch.einsum('bd,bdn->bn', text, flat).reshape(b, 1, side, side)
    sim = F.interpolate(sim, size=(size, size), mode='bilinear', align_corners=False)
    return sim[:, 0].cpu().numpy()

heatmaps = e1_heatmap(rows)
print('heatmaps:', heatmaps.shape, '| min/max:', float(heatmaps.min()), float(heatmaps.max()))

In [ ]:
import pandas as pd

def norm01(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def iou(pred, gt, thr=0.5):
    p, g = pred >= thr, gt >= 0.5
    union = (p | g).sum()
    return float((p & g).sum() / union) if union else float('nan')

def nss(pred, gt):
    z = (pred - pred.mean()) / (pred.std() + 1e-8)
    m = gt >= 0.5
    return float((z * m).sum() / max(m.sum(), 1))

results = []
for row, hm in zip(rows, heatmaps):
    _, mask = load_sample(row)
    p = norm01(hm)
    results.append({'object': row['object'], 'affordance': row['affordance'],
                    'iou': iou(p, mask), 'nss': nss(p, mask)})

df = pd.DataFrame(results)
display(df)
print('mean IoU:', round(df.iou.mean(), 4), '| mean NSS:', round(df.nss.mean(), 4))
df.to_csv('smoke_e1_metrics.csv', index=False)

In [ ]:
fig, axes = plt.subplots(len(rows), 3, figsize=(9, 3 * len(rows)))
for i, (row, hm) in enumerate(zip(rows, heatmaps)):
    img, mask = load_sample(row)
    p = norm01(hm)
    axes[i, 0].imshow(img); axes[i, 0].set_title(f"{row['object']} / {row['affordance']}", fontsize=9)
    axes[i, 1].imshow(mask, cmap='gray'); axes[i, 1].set_title('GT mask', fontsize=9)
    axes[i, 2].imshow(img); axes[i, 2].imshow(p, cmap='jet', alpha=0.5)
    axes[i, 2].set_title(f"E1 IoU={iou(p, mask):.2f}", fontsize=9)
    for ax in axes[i]:
        ax.axis('off')
plt.tight_layout()
plt.savefig('smoke_e1_heatmaps.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved: smoke_e1_heatmaps.png, smoke_e1_metrics.csv')

## 怎么判断冒烟是否通过

| 检查项 | 通过标准 |
|---|---|
| CLIP 加载 | 打印出 patch grid，形状类似 `(1, 768, 14, 14)`，side=14 |
| E1 前向 | 5 张图都输出 224×224 热图，无报错 |
| 指标 | `mean IoU` / `mean NSS` 是数字（不全是 NaN） |
| 可视化 | 第三列热图大体落在物体附近 |

> E1 是**弱基线**，IoU 偏低是正常的。这一步只证明代码通路可用，不代表方法效果。
> 跑完后点 **Save Version**，产物会保留在 `/kaggle/working`。

## 下一步

- **Phase 1（12-20 之后）**：下载 AGD20K → 256px 紧凑版 → 上传私有 Dataset；
- **Phase 2**：E2（冻结 CLIP + FiLM 解码器）训练与三划分评测；
- 正式实验前，把本 Notebook 的随机划分换成 AGD20K 官方对象类别协议。